# 🧬 Hydra-PaiNN: Aqueous Tautomer Ratio & Free Energy Predictor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeantChen/TautomerRatio/blob/main/notebooks/Hydra_PaiNN_Quickstart.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue.svg?logo=github)](https://github.com/KeantChen/TautomerRatio)

**Hydra-PaiNN** predicts aqueous tautomer equilibrium constants ($K_\text{eq}$, $\log_{10} K_\text{eq}$), population percentages, and relative free energies ($\Delta G_\text{aq} = G_B - G_A = -RT \ln K_\text{eq}$).

### 🎯 Applicability Domains (3-Tier Classification):
- **Comfort Zone** ($|\Delta N_\text{aro}| = 0$, 79.4% of dataset):
  - **Tier 1 — CBH-Taut Subset**: Invariant heavy-atom hybridizations and bond types. Expected MAE: **1.29 kcal/mol**.
  - **Tier 2 — Normal Subset**: Aromaticity conserved with localized bond rearrangements. Expected MAE: **1.97 kcal/mol**.
- **Risk Zone** ($|\Delta N_\text{aro}| > 0$, 20.6% of dataset):
  - **Tier 3 — Risk Zone**: Aromaticity transitions. Expected MAE: **5.14 kcal/mol** (DFT reference MAE = 4.07 kcal/mol). *(Update is coming soon)*


In [ ]:
#@title 1. Environment Setup & Dependency Installation
#@markdown Run this cell once to clone/update the repository, install Python packages, and configure Linux xTB 6.7.1.
import os, sys, shutil, glob, importlib

# 1. Clone repository into Colab workspace (or force-sync to latest commit if already present)
repo_dir = '/content/TautomerRatio'
if not os.path.exists(repo_dir):
    print('📥 Cloning Hydra-PaiNN repository from GitHub...')
    !git clone https://github.com/KeantChen/TautomerRatio.git /content/TautomerRatio
else:
    print('🔄 Syncing repository to latest commit from GitHub...')
    !cd /content/TautomerRatio && git fetch origin && git reset --hard origin/main

%cd /content/TautomerRatio
if '/content/TautomerRatio' not in sys.path:
    sys.path.insert(0, '/content/TautomerRatio')

# Clear any cached module in memory so updates take effect immediately without restarting kernel
for mod_name in list(sys.modules.keys()):
    if 'tautomer_predictor' in mod_name:
        del sys.modules[mod_name]

# 2. Install Python dependencies
print('📦 Installing RDKit and PyTorch Geometric...')
!pip install -q rdkit torch_geometric

# 3. Download and set up Linux xTB binary
xtb_target = '/content/TautomerRatio/bin/xtb'
if not os.path.exists(xtb_target) or os.path.getsize(xtb_target) < 1000000:
    print('⚙️ Downloading and setting up precompiled xTB 6.7.1 for Linux x86_64...')
    if not os.path.exists('/content/xtb-dist/bin/xtb'):
        !wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz -O /content/xtb.tar.xz
        !tar -xf /content/xtb.tar.xz -C /content/
    
    os.makedirs('/content/TautomerRatio/bin', exist_ok=True)
    found_bins = [p for p in glob.glob('/content/**/bin/xtb', recursive=True) if not p.startswith('/content/TautomerRatio/bin')]
    if found_bins:
        src_xtb = found_bins[0]
        shutil.copy2(src_xtb, xtb_target)
        os.chmod(xtb_target, 0o755)
        try:
            shutil.copy2(src_xtb, '/usr/local/bin/xtb')
            os.chmod('/usr/local/bin/xtb', 0o755)
        except Exception:
            pass
        print(f'✅ Installed xTB binary from {src_xtb} -> {xtb_target}')
    else:
        !cp -f /content/xtb-dist/bin/xtb /content/TautomerRatio/bin/xtb
        !chmod +x /content/TautomerRatio/bin/xtb

# 4. Runtime environment configuration for xTB
share_dirs = glob.glob('/content/**/share/xtb', recursive=True)
xtb_share = share_dirs[0] if share_dirs else '/content/xtb-dist/share/xtb'

os.environ['XTBPATH'] = xtb_share
os.environ['PATH'] = '/content/TautomerRatio/bin:/content/xtb-dist/bin:' + os.environ.get('PATH', '')
os.environ['OMP_STACKSIZE'] = '2G'
os.environ['OMP_MAX_ACTIVE_LEVELS'] = '1'

print('⚙️ Verifying xTB binary execution directly...')
!/content/TautomerRatio/bin/xtb --version

print('✅ Environment setup successfully completed! You are ready to run predictions.')


In [ ]:
#@title 2. Verify Installation & Model Loading
#@markdown Runs self-test diagnostic: PyTorch, RDKit, 3-Tier triage, weights, scalers, and end-to-end inference.
!python verify_installation.py


In [ ]:
#@title 3. Interactive Tautomer Prediction & Structure Visualization
#@markdown Enter two tautomeric SMILES representations (Note: in TautoBase, $\Delta G_\text{exp} = -1.364 \times \log_{10} K_\text{exp}$ at 298.15 K):
smiles_A = 'OC1=CC=CC=N1'  #@param {type:'string'}
smiles_B = 'O=C1NC=CC=C1'  #@param {type:'string'}
solvent = 'water'  #@param ['water', 'gas']
temperature_K = 298.15  #@param {type:'number'}

import importlib
from IPython.display import display, HTML
from rdkit import Chem
from rdkit.Chem import Draw
import tautomer_predictor
importlib.reload(tautomer_predictor)
from tautomer_predictor import TautomerPredictor

predictor = TautomerPredictor()

print(f'🔬 Running Hydra-PaiNN prediction (GFN2-xTB opt+hess + PaiNN):')
print(f'   Tautomer A: {smiles_A}')
print(f'   Tautomer B: {smiles_B}\n')

res = predictor.predict(smiles_A, smiles_B, temperature=temperature_K, solvent=solvent)
triage = res['triage']
pred = res['recommended_prediction']
subset_label = triage.get('subset', triage.get('tier_name', ''))

# 2D Molecular Visualization
mol_A = Chem.MolFromSmiles(smiles_A)
mol_B = Chem.MolFromSmiles(smiles_B)
legends = [
    f"Tautomer A ({pred['population_pct']['A']:.1f}%)" + (" [Major]" if pred['major_tautomer'] == 'A' else ''),
    f"Tautomer B ({pred['population_pct']['B']:.1f}%)" + (" [Major]" if pred['major_tautomer'] == 'B' else '')
]
img = Draw.MolsToGridImage([mol_A, mol_B], molsPerRow=2, subImgSize=(320, 200), legends=legends)
display(img)

# Summary Card
tier_colors = {1: '#10b981', 2: '#3b82f6', 3: '#ef4444'}
tier_color = tier_colors.get(triage['tier'], '#6b7280')
log10_k = pred.get('log10_K', 0.0)

html_card = f"""
<div style='background-color: #f8fafc; border: 1px solid #e2e8f0; border-radius: 12px; padding: 20px; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; max-width: 680px; margin-top: 15px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);'>
  <div style='display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #e2e8f0; padding-bottom: 12px; margin-bottom: 16px;'>
    <h3 style='margin: 0; color: #1e293b; font-size: 18px;'>Prediction Summary</h3>
    <span style='background-color: {tier_color}; color: white; padding: 4px 12px; border-radius: 9999px; font-size: 12px; font-weight: bold;'>
      Tier {triage['tier']}: {subset_label}
    </span>
  </div>
  
  <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;'>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Relative Free Energy (ΔG_aq = G_B - G_A)</div>
      <div style='font-size: 20px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['delta_G_kcal_mol']:+.2f} <span style='font-size: 13px; font-weight: normal; color: #64748b;'>kcal/mol</span></div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>log10(K_eq) / K_eq ([B]/[A])</div>
      <div style='font-size: 20px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{log10_k:+.2f} <span style='font-size: 13px; font-weight: normal; color: #64748b;'>({pred['K_eq']:.2e})</span></div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Major Tautomer</div>
      <div style='font-size: 20px; font-weight: bold; color: #2563eb; margin-top: 4px;'>Tautomer {pred['major_tautomer']}</div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Boltzmann Ratio</div>
      <div style='font-size: 18px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['ratio']}</div>
    </div>
  </div>

  <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7; font-size: 13px; color: #334155; line-height: 1.5;'>
    <strong>Domain & Subset:</strong> {triage.get('domain', 'Comfort Zone')} &rarr; {subset_label}<br>
    <strong>Expected Benchmark MAE:</strong> ±{triage['expected_mae']:.2f} kcal/mol<br>
    <strong>Diagnostic:</strong> {triage['message']}
  </div>
</div>
"""
display(HTML(html_card))


In [ ]:
#@title 4. Batch Prediction & Ground-Truth Comparison on Example Benchmark CSV
#@markdown Runs predictions on `examples/test_cases.csv` (12 benchmark pairs across Tiers 1, 2, and 3) and compares against experimental $\Delta G_\text{exp}$ and $\log_{10} K_\text{exp}$.
import pandas as pd
from IPython.display import display

!python predict.py --input examples/test_cases.csv --output colab_predictions.csv --mode xtb_only

df_out = pd.read_csv('colab_predictions.csv')
display_cols = [
    c for c in [
        'id', 'name', 'triage_tier', 'subset', 'pred_dG_kcal_mol', 'dG_exp',
        'error_abs_kcal', 'pred_log10_K', 'logK_exp', 'major_tautomer', 'exp_major',
        'pop_A_pct', 'pop_B_pct'
    ] if c in df_out.columns
]
print(f'✅ Batch prediction completed! Showing comparison with experimental ground truth ({len(df_out)} pairs):')
display(df_out[display_cols])


In [ ]:
#@title 5. High-Throughput 2D Topological Triage Screening (<1 ms per pair)
#@markdown Rapidly screen tautomer pairs based on aromaticity conservation and Raghavachari CBH rules without running 3D quantum calculations.
import importlib
import pandas as pd
import tautomer_predictor
importlib.reload(tautomer_predictor)
from tautomer_predictor import TautomerPredictor

df_cases = pd.read_csv('examples/test_cases.csv')
print(f"{'ID':<9} {'Name':<35} {'Tier':<8} {'Subset':<18} {'Domain':<15} {'Expected MAE':<15}")
print('-' * 102)
for _, row in df_cases.iterrows():
    triage = TautomerPredictor.triage_check(row['tautomer1'], row['tautomer2'])
    subset_str = triage.get('subset', triage.get('tier_name', ''))
    print(f"{row['id']:<9} {str(row['name'])[:33]:<35} Tier {triage['tier']:<3} {subset_str:<18} {triage['domain']:<15} ±{triage['expected_mae']:.2f} kcal/mol")
